In [17]:
from ase import atoms
from rdkit.Chem import AllChem
import numpy as np
from rdkit2ase import rdkit2ase
from ase.calculators.lj import LennardJones
from ase.optimize import BFGS
from ase.visualize import view
from ase import units
from rdkit.Chem import rdMolAlign
from rdkit import Chem

Molecule setup

In [18]:
input = "CC(C)CC(C)CO"

mol = AllChem.MolFromSmiles(input)
mol = AllChem.AddHs(mol)



2. conformer generation
3. Geometry optimization
4. save optimized structures
- initialize report

In [ ]:
# generate + add conformers to the molecule object
# *** FIND WAY TO DETERMINE NUM CONFORMERS BASED ON MOLECULE SIZE/COMPLEXITY
NUM_CONFS = 10
AllChem.EmbedMultipleConfs(mol, numConfs=NUM_CONFS)


# find proper optimization value
F_MAX = 0.05

results = []

for conf in mol.GetConformers():
    conf_id = conf.GetId()

    # construct ASE atoms object from conformer
    positions = conf.GetPositions()
    symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
    atom = atoms.Atoms(symbols=symbols, positions=positions)

    atom.calc = LennardJones()
    opt = BFGS(atom)
    opt.run(fmax=F_MAX)

# update rdkit conformer positions with optimized positions from ASE
    optimized_positions = atom.get_positions()
for idx in range(mol.GetNumAtoms()):
    x, y, z = optimized_positions[idx]
    conf.SetAtomPosition(idx, (float(x), float(y), float(z)))

    # create list of "results" dict, contains each conformer profile
    results.append({
        "id": conf_id,
        "energy": atom.get_potential_energy() / (units.kcal/units.mol), # stores energy in eV --> convert to kcal/mol
        "conformer": conf, # rdkit conformer object
        "atoms": atom.copy() # ASE atoms object for each conformer
    })

results.sort(key=lambda x: x["energy"])
print(f"Number of prefiltered conformers: {len(results)}")

      Step     Time          Energy          fmax
BFGS:    0 18:32:05      -19.610140       30.479404
BFGS:    1 18:32:05      -20.820638        2.776417
BFGS:    2 18:32:05      -21.273825        2.841101
BFGS:    3 18:32:05      -16.146680      129.475868
BFGS:    4 18:32:05      -22.797447        2.784522
BFGS:    5 18:32:05      -24.360890        3.157697
BFGS:    6 18:32:05      -21.428792       52.201329
BFGS:    7 18:32:05      -26.533125        6.611713
BFGS:    8 18:32:05      -27.122985       13.306579
BFGS:    9 18:32:05      -30.313444        8.694172
BFGS:   10 18:32:05      -32.278985       31.577815
BFGS:   11 18:32:05      -28.172591      103.556523
BFGS:   12 18:32:05       -0.922495      591.925875
BFGS:   13 18:32:05      -19.048514      115.671144
BFGS:   14 18:32:05      -32.343769       49.177477
BFGS:   15 18:32:05      -34.774396       28.407258
BFGS:   16 18:32:05      -36.159212       29.704102
BFGS:   17 18:32:05      -28.684124      151.637667
BFGS:   18 18:

5. Energy filtering
- remove all optimized conformers 5 kcal/mol above minimum value, not likely to be physically formed

In [ ]:
min_energy = results[0]["energy"]
ENERGY_THRESHOLD = 5.0 # kcal/mol
conformer_list_filtered = []

for result in results:
    if result["energy"] - min_energy <= ENERGY_THRESHOLD:
        conformer_list_filtered.append(result)

conformer_list_filtered.sort(key=lambda x: x["energy"])

print(f"Number of filtered conformers: {len(conformer_list_filtered)}")
conformer_list_filtered

Number of filtered conformers: 2


[{'conformer': <rdkit.Chem.rdchem.Conformer at 0x2d9d0011fc0>,
  'energy': np.float64(-2098.9573486329255),
  'conf_id': 9,
  'atoms': Atoms(symbols='C7OH16', pbc=False),
  'positions': array([[ 0.60085259, -0.00945834, -0.05058883],
         [ 0.04046903,  0.42543592,  0.78755478],
         [ 1.6626942 ,  0.09263968, -0.09154592],
         [ 0.19424012, -0.96615495, -0.40123225],
         [-0.73179343, -1.14394485,  0.22559679],
         [-1.47694399, -0.31403135, -0.01046238],
         [-0.42573139, -0.14532881, -0.02508174],
         [-0.04238383,  0.85100984, -0.2779217 ],
         [ 0.95767164,  0.66831761, -0.79953687],
         [ 1.08732954, -0.46753601, -0.89322166],
         [ 1.81743777,  1.17144143, -0.2843405 ],
         [ 1.16959124, -0.89414228,  0.17420758],
         [ 1.08921171, -0.01140248,  0.90288322],
         [ 1.89830986,  0.75296794,  0.76483091],
         [ 0.91021116,  0.95112252,  0.30886386],
         [ 0.18715251, -0.68832473,  0.70504806],
         [-0.851

6. RMSD analysis
- pairwise comparisons of each conformer to determine similarity

In [ ]:
# rdMolAlign.GetBestRMS() only takes molecule objects and considers conformers by their IDs
# must create a template molecule with newly filtered conformers
template_mol = Chem.Mol(mol)
template_mol.RemoveAllConformers()
new_ids = []
for item in conformer_list_filtered:
    # Adding the conformer object back to the template
    # assignId=True ensures they get fresh ids
    new_id = template_mol.AddConformer(item["conformer"], assignId=True)
    new_ids.append(new_id)

# create 0s matrix to store rmsd values
n_conf = len(conformer_list_filtered)

rmsd_matrix = np.zeros((n_conf, n_conf))

# fill in rms values by comparing each conformer to each other conformer (only upper triangle since symmetric)
for i in range(n_conf):
    for j in range(i + 1, n_conf):
        rmsd_matrix[i, j] = rdMolAlign.GetBestRMS(
            template_mol, 
            template_mol, 
            prbId=new_ids[i], 
            refId=new_ids[j]
        )
        rmsd_matrix[j, i] = rmsd_matrix[i, j]

RMSD_THRESHOLD = 0.75 # in angstroms, upper threshold for how similar two conformers must be to be considered duplicates
# can be ambiguous/tuned based on molecule size/complexity...maybe implement same algorithm as NUM_CONFS to determine this value based on size + complexity?

rmsd_matrix

# earlier conformer generation is not deterministic, so there can be different outcomes for filtered conformers
# implement random seed for debugging if needed

array([[0.        , 1.51884235],
       [1.51884235, 0.        ]])

In [ ]:

'''
# generate ase Atoms object for more optimal conformer and view
genpositions = results[0]['conformer'].GetPositions()
gensymbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
genatom = atoms.Atoms(symbols=gensymbols, positions=genpositions)
view(genatom)
'''

<Popen: returncode: None args: ['c:\\Users\\avirn\\Dhruvcoding\\ase\\.venv\\...>